> **Note**: This notebook performs comprehensive dataset cleaning and validation for the general Waste Detection YOLO pipeline. It reads class definitions dynamically from `data.yaml` — no hardcoded class lists.

# 🧹 Waste Detection — Dataset Cleaning & Validation (Notebook 04)

### Overview
This notebook cleans and validates the YOLO dataset produced by Notebook 03 (COCO → YOLO Conversion).

### Operations
1. Load and validate dataset configuration from `data.yaml`
2. Check image integrity (corrupted, zero-byte, unsupported formats)
3. Validate label files (format, class IDs, bounding boxes)
4. Fix out-of-bounds coordinates, remove invalid entries
5. Detect and remove duplicate images (MD5 hash)
6. Remove duplicate label lines
7. Generate cleaned dataset with proper structure
8. Produce quality reports

### Pipeline Position
```
NB 03 (COCO→YOLO) → [taco_yolo_all_categories/] → NB 04 (THIS) → [taco_yolo_cleaned/] → NB 05
```

## 1. Environment Setup & Library Imports

In [ ]:
!pip install -q rich tqdm pyyaml pandas opencv-python matplotlib pillow scikit-learn

In [ ]:
import os
import json
import shutil
import hashlib
import random
import math
import warnings
from pathlib import Path
from typing import List, Dict, Set, Tuple, Any

import yaml
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()
warnings.filterwarnings('ignore')
random.seed(42)

## 2. Load Dataset Configuration
Load `data.yaml` from the NB 03 output. Class names and IDs are read **dynamically** — never hardcoded.

In [ ]:
# ==========================================
# Configuration — Adjust paths as needed
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')

# Input: Output of NB 03 (COCO → YOLO Conversion)
INPUT_YOLO_DIR = PROJECT_ROOT / 'datasets/taco_yolo_all_categories'
YAML_PATH = INPUT_YOLO_DIR / 'data.yaml'

# Output: Cleaned dataset
OUTPUT_YOLO_DIR = PROJECT_ROOT / 'datasets/taco_yolo_cleaned'
REPORTS_DIR = OUTPUT_YOLO_DIR / 'reports'

# ==========================================
# Load and validate data.yaml
# ==========================================
if not YAML_PATH.exists():
    raise FileNotFoundError(
        f"data.yaml not found at {YAML_PATH}. "
        f"Run Notebook 03 (COCO → YOLO Conversion) first."
    )

with open(YAML_PATH, 'r') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config.get('names', {})
# Ensure keys are ints
CLASS_NAMES = {int(k): v for k, v in CLASS_NAMES.items()}
VALID_CLASS_IDS = set(CLASS_NAMES.keys())
NUM_CLASSES = len(CLASS_NAMES)

console.print(f"[green]✔ Loaded data.yaml — {NUM_CLASSES} classes[/green]")
console.print(f"[cyan]  Input:  {INPUT_YOLO_DIR}[/cyan]")
console.print(f"[cyan]  Output: {OUTPUT_YOLO_DIR}[/cyan]")

# Quick sanity check
expected_nc = config.get('nc', NUM_CLASSES)
assert NUM_CLASSES == expected_nc, f"nc mismatch: {NUM_CLASSES} names vs nc={expected_nc}"

# Print all classes
table = Table(title=f"Dataset Classes ({NUM_CLASSES} total)", show_header=True, show_lines=False)
table.add_column("ID", style="cyan", justify="right")
table.add_column("Name", style="white")
for cid in sorted(CLASS_NAMES.keys()):
    table.add_row(str(cid), CLASS_NAMES[cid])
console.print(table)

## 3. Dataset Cleaning Engine
Comprehensive cleaning: image validation, label validation, deduplication, coordinate fixing.

In [ ]:
class DatasetCleaner:
    """Comprehensive YOLO dataset cleaner with detailed reporting."""

    def __init__(self, src_dir: Path, dest_dir: Path, valid_ids: Set[int]):
        self.src_dir = src_dir
        self.dest_dir = dest_dir
        self.valid_ids = valid_ids
        self.image_hashes: Set[str] = set()

        self.stats = {
            'total_processed': 0,
            'missing_images': 0,
            'missing_labels': 0,
            'empty_labels': 0,
            'invalid_class_ids': 0,
            'invalid_bboxes': 0,
            'out_of_bounds_bboxes': 0,
            'duplicate_labels': 0,
            'corrupted_images': 0,
            'zero_byte_images': 0,
            'duplicate_images': 0,
            'unsupported_formats': 0,
            'images_removed': 0,
            'labels_fixed': 0,
            'images_kept': 0,
            'annotations_kept': 0,
        }

    def is_valid_image(self, img_path: Path) -> bool:
        """Check if an image file is valid."""
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            self.stats['unsupported_formats'] += 1
            return False
        if img_path.stat().st_size == 0:
            self.stats['zero_byte_images'] += 1
            return False
        try:
            with Image.open(img_path) as img:
                img.verify()
            # Re-open to check actual readability
            with Image.open(img_path) as img:
                img.load()
            return True
        except Exception:
            self.stats['corrupted_images'] += 1
            return False

    def is_duplicate(self, img_path: Path) -> bool:
        """Check for duplicate images using MD5 hash."""
        hasher = hashlib.md5()
        with open(img_path, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                hasher.update(chunk)
        img_hash = hasher.hexdigest()
        if img_hash in self.image_hashes:
            self.stats['duplicate_images'] += 1
            return True
        self.image_hashes.add(img_hash)
        return False

    def clean_label(self, label_path: Path) -> List[str]:
        """Validate and fix a YOLO label file. Returns cleaned lines."""
        if not label_path.exists():
            self.stats['missing_labels'] += 1
            return []

        with open(label_path, 'r') as f:
            lines = f.readlines()

        if not lines:
            self.stats['empty_labels'] += 1
            return []

        cleaned = []
        seen_lines = set()

        for line in lines:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) != 5:
                self.stats['invalid_bboxes'] += 1
                continue

            try:
                cls_id = int(parts[0])
                x_c, y_c, w, h = [float(p) for p in parts[1:]]
            except ValueError:
                self.stats['invalid_bboxes'] += 1
                continue

            # Validate class ID
            if cls_id not in self.valid_ids:
                self.stats['invalid_class_ids'] += 1
                continue

            # Fix out-of-bounds coordinates
            fixed = False
            if x_c < 0 or x_c > 1 or y_c < 0 or y_c > 1 or w <= 0 or w > 1 or h <= 0 or h > 1:
                # Attempt to clip
                x_c = max(0.0, min(1.0, x_c))
                y_c = max(0.0, min(1.0, y_c))
                w = max(0.001, min(1.0, w))
                h = max(0.001, min(1.0, h))

                # Check if box is still valid after clipping
                if (x_c - w/2) < -0.01 or (x_c + w/2) > 1.01 or (y_c - h/2) < -0.01 or (y_c + h/2) > 1.01:
                    self.stats['out_of_bounds_bboxes'] += 1
                    continue

                self.stats['labels_fixed'] += 1
                fixed = True

            clean_line = f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}"

            # Deduplicate labels within same file
            if clean_line in seen_lines:
                self.stats['duplicate_labels'] += 1
                continue
            seen_lines.add(clean_line)
            cleaned.append(clean_line)

        return cleaned

    def clean_split(self, split: str):
        """Clean a single split (train/val/test)."""
        src_img_dir = self.src_dir / 'images' / split
        src_lbl_dir = self.src_dir / 'labels' / split
        dst_img_dir = self.dest_dir / 'images' / split
        dst_lbl_dir = self.dest_dir / 'labels' / split

        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        if not src_img_dir.exists():
            console.print(f"[yellow]⚠ Source image dir missing: {src_img_dir}[/yellow]")
            return

        # Get all images
        images = sorted(
            list(src_img_dir.glob('*.[jJ][pP][gG]')) +
            list(src_img_dir.glob('*.[jJ][pP][eE][gG]')) +
            list(src_img_dir.glob('*.[pP][nN][gG]'))
        )

        for img_path in tqdm(images, desc=f"Cleaning {split}"):
            self.stats['total_processed'] += 1

            # Check image validity
            if not self.is_valid_image(img_path):
                self.stats['images_removed'] += 1
                continue

            # Check for duplicates
            if self.is_duplicate(img_path):
                self.stats['images_removed'] += 1
                continue

            # Clean label
            lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
            cleaned_lines = self.clean_label(lbl_path)

            # Copy image
            dst_img = dst_img_dir / img_path.name
            shutil.copy2(str(img_path), str(dst_img))

            # Write cleaned label (may be empty for negative examples)
            dst_lbl = dst_lbl_dir / f"{img_path.stem}.txt"
            with open(dst_lbl, 'w') as f:
                if cleaned_lines:
                    f.write("\n".join(cleaned_lines))

            self.stats['images_kept'] += 1
            self.stats['annotations_kept'] += len(cleaned_lines)

        console.print(f"[green]  ✔ {split}: {len(images)} → {self.stats['images_kept']} images[/green]")

In [ ]:
# ==========================================
# Execute cleaning
# ==========================================
# Clear output directory
if OUTPUT_YOLO_DIR.exists():
    shutil.rmtree(OUTPUT_YOLO_DIR)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

cleaner = DatasetCleaner(INPUT_YOLO_DIR, OUTPUT_YOLO_DIR, VALID_CLASS_IDS)

console.print("[cyan]Starting dataset cleaning...[/cyan]")
for split in ['train', 'val', 'test']:
    cleaner.clean_split(split)

cleaning_stats = cleaner.stats

console.print(Panel.fit(
    f"[bold cyan]Cleaning Summary[/bold cyan]\n\n"
    f"Total processed:    {cleaning_stats['total_processed']}\n"
    f"Images kept:        {cleaning_stats['images_kept']}\n"
    f"Images removed:     {cleaning_stats['images_removed']}\n"
    f"Annotations kept:   {cleaning_stats['annotations_kept']}\n"
    f"Labels fixed:       {cleaning_stats['labels_fixed']}\n"
    f"Corrupted images:   {cleaning_stats['corrupted_images']}\n"
    f"Duplicate images:   {cleaning_stats['duplicate_images']}\n"
    f"Invalid class IDs:  {cleaning_stats['invalid_class_ids']}\n"
    f"Invalid bboxes:     {cleaning_stats['invalid_bboxes']}\n"
    f"OOB bboxes:         {cleaning_stats['out_of_bounds_bboxes']}\n"
    f"Duplicate labels:   {cleaning_stats['duplicate_labels']}"
))

## 4. Dataset Quality Analysis
Analyze the cleaned dataset and compute quality metrics.

In [ ]:
# ==========================================
# Analyze cleaned dataset quality
# ==========================================
def analyze_dataset_quality(clean_dir: Path, original_stats: dict) -> dict:
    splits = ['train', 'val', 'test']
    split_counts = {s: 0 for s in splits}
    class_counts = {cid: 0 for cid in VALID_CLASS_IDS}
    total_objs = 0

    for split in splits:
        lbl_dir = clean_dir / 'labels' / split
        if not lbl_dir.exists():
            continue

        labels = list(lbl_dir.glob('*.txt'))
        split_counts[split] = len(labels)

        for lbl in labels:
            with open(lbl, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    cls_id = int(line.split()[0])
                    if cls_id in class_counts:
                        class_counts[cls_id] += 1
                    total_objs += 1

    total_images = sum(split_counts.values())

    # Quality Score
    error_count = original_stats['images_removed'] + original_stats['labels_fixed']
    error_rate = error_count / max(1, original_stats['total_processed'])
    quality_score = max(0.0, min(100.0, (1.0 - error_rate) * 100))

    quality_metrics = {
        'Total Clean Images': total_images,
        'Total Clean Objects': total_objs,
        'Train Split': split_counts['train'],
        'Val Split': split_counts['val'],
        'Test Split': split_counts['test'],
        'Overall Quality Score': round(quality_score, 2)
    }

    # Per-class distribution
    table = Table(title='Class Distribution (Cleaned Dataset)', show_header=True, show_lines=False)
    table.add_column('ID', style='cyan', justify='right')
    table.add_column('Class Name', style='white')
    table.add_column('Count', justify='right')
    table.add_column('% of Total', justify='right')

    for cid in sorted(class_counts.keys()):
        count = class_counts[cid]
        pct = (count / total_objs * 100) if total_objs > 0 else 0
        table.add_row(str(cid), CLASS_NAMES[cid], str(count), f'{pct:.1f}%')

    console.print(table)

    console.print(Panel.fit(
        f"[bold cyan]Quality Score: {quality_score:.1f}%[/bold cyan]\n\n"
        f"Clean Images: {total_images}\n"
        f"Clean Objects: {total_objs}\n"
        f"Train: {split_counts['train']} | Val: {split_counts['val']} | Test: {split_counts['test']}"
    ))

    return quality_metrics, class_counts

quality_metrics, class_distribution = analyze_dataset_quality(OUTPUT_YOLO_DIR, cleaning_stats)

## 5. Visual Verification
Visualize random cleaned images with their YOLO annotations to verify correctness.

In [ ]:
def visualize_cleaned(yolo_dir: Path, class_names: dict, num_images: int = 8):
    """Visualize random images with their YOLO annotations."""
    train_imgs = sorted(
        list((yolo_dir / 'images' / 'train').glob('*.[jJ][pP][gG]')) +
        list((yolo_dir / 'images' / 'train').glob('*.[pP][nN][gG]'))
    )
    if not train_imgs:
        console.print("[yellow]No images found for visualization.[/yellow]")
        return

    samples = random.sample(train_imgs, min(num_images, len(train_imgs)))
    num_classes = len(class_names)

    # Generate deterministic colors
    np.random.seed(42)
    colors = {}
    for cls_id in range(num_classes):
        hue = int(cls_id * 180 / max(num_classes, 1)) % 180
        hsv = np.array([[[hue, 200, 230]]], dtype=np.uint8)
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0][0]
        colors[cls_id] = (int(bgr[2]), int(bgr[1]), int(bgr[0]))  # RGB

    cols = 4
    rows = math.ceil(len(samples) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    if rows * cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, img_path in enumerate(samples):
        lbl_path = yolo_dir / 'labels' / 'train' / f"{img_path.stem}.txt"
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_h, img_w = image.shape[:2]

        ann_count = 0
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        x_c, y_c, w, h = [float(p) for p in parts[1:]]
                        abs_w, abs_h = int(w * img_w), int(h * img_h)
                        abs_x = int((x_c * img_w) - (abs_w / 2))
                        abs_y = int((y_c * img_h) - (abs_h / 2))
                        color = colors.get(cls_id, (255, 0, 0))
                        cv2.rectangle(image, (abs_x, abs_y), (abs_x+abs_w, abs_y+abs_h), color, 2)
                        label = f"[{cls_id}] {class_names.get(cls_id, '?')}"
                        cv2.putText(image, label, (abs_x, max(12, abs_y-4)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
                        ann_count += 1

        axes[i].imshow(image)
        axes[i].set_title(f"{img_path.name} ({ann_count} objs)", fontsize=9)
        axes[i].axis('off')

    for j in range(len(samples), len(axes)):
        axes[j].axis('off')

    plt.suptitle('Cleaned Dataset — Visual Verification', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_cleaned(OUTPUT_YOLO_DIR, CLASS_NAMES, num_images=8)

## 6. Generate Cleaned `data.yaml`
Create a new `data.yaml` for the cleaned dataset with updated paths.

In [ ]:
# ==========================================
# Generate data.yaml for cleaned dataset
# ==========================================
clean_yaml = {
    'path': str(OUTPUT_YOLO_DIR.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES
}

clean_yaml_path = OUTPUT_YOLO_DIR / 'data.yaml'
with open(clean_yaml_path, 'w') as f:
    yaml.dump(clean_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

console.print(f"[green]✔ Generated data.yaml for cleaned dataset[/green]")
console.print(f"[cyan]  nc: {NUM_CLASSES}[/cyan]")
console.print(f"[cyan]  Path: {clean_yaml_path}[/cyan]")

## 7. Report Generation
Export cleaning and quality reports.

In [ ]:
# ==========================================
# Export reports
# ==========================================
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Cleaning Report
with open(REPORTS_DIR / 'cleaning_report.json', 'w') as f:
    json.dump(cleaning_stats, f, indent=4)
pd.DataFrame([cleaning_stats]).to_csv(REPORTS_DIR / 'cleaning_report.csv', index=False)

# Quality Report
with open(REPORTS_DIR / 'dataset_quality.json', 'w') as f:
    json.dump(quality_metrics, f, indent=4)
pd.DataFrame([quality_metrics]).to_csv(REPORTS_DIR / 'dataset_quality.csv', index=False)

# Annotation validation report
validation_data = {**cleaning_stats, **quality_metrics}
with open(REPORTS_DIR / 'annotation_validation_report.json', 'w') as f:
    json.dump(validation_data, f, indent=4)
pd.DataFrame([validation_data]).to_csv(REPORTS_DIR / 'annotation_validation_report.csv', index=False)

console.print(f"[green]✔ Reports generated in {REPORTS_DIR}[/green]")

## 8. Final Summary

In [ ]:
console.print(Panel.fit(
    f"[bold green]Dataset Cleaned & Validated[/bold green]\n\n"
    f"[bold cyan]✔ Quality Score:[/bold cyan] {quality_metrics['Overall Quality Score']}%\n"
    f"[bold red]✖ Images Removed:[/bold red] {cleaning_stats['images_removed']}\n"
    f"[bold yellow]🔧 Labels Fixed:[/bold yellow] {cleaning_stats['labels_fixed']}\n"
    f"[bold red]✖ Duplicate Labels:[/bold red] {cleaning_stats['duplicate_labels']}\n"
    f"[bold cyan]✔ Final Dataset:[/bold cyan] {quality_metrics['Total Clean Images']} images, "
    f"{quality_metrics['Total Clean Objects']} objects\n"
    f"[bold cyan]✔ Classes:[/bold cyan] {NUM_CLASSES}\n\n"
    f"[bold]Input:[/bold]  {INPUT_YOLO_DIR}\n"
    f"[bold]Output:[/bold] {OUTPUT_YOLO_DIR}\n\n"
    f"[bold magenta]Next Notebook:[/bold magenta] 05_Data_Augmentation.ipynb"
))